In [6]:
# ==============================================================================
# SETUP & DEPENDENCIES (Free Google Gemini API)
# ==============================================================================
# Run this in a Colab cell first:
# !pip install -q google-genai pydantic

import os
import sqlite3
import json
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Use Colab's built-in Secret Manager or fallback to environment variables
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
except ImportError:
    GEMINI_API_KEY = os.environ.get('GOOGLE_API_KEY', 'YOUR_API_KEY_HERE')

# FIX: Change 'GOOGLE_API_KEY' to 'GEMINI_API_KEY' here
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = 'gemini-3.6-flash' # Free tier compatible

# ==============================================================================
# DATABASE ENVIRONMENT SETUP
# ==============================================================================
def setup_ecommerce_db():
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE customers (
            id INTEGER PRIMARY KEY,
            name TEXT,
            tier TEXT,
            spent REAL
        )
    """)
    cursor.execute("""
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            status TEXT,
            total REAL,
            FOREIGN KEY(customer_id) REFERENCES customers(id)
        )
    """)
    cursor.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", [
        (1, 'Alice Corp', 'Enterprise', 15400.0),
        (2, 'Bob LLC', 'Standard', 1200.5),
        (3, 'Charlie Inc', 'Enterprise', 8900.0)
    ])
    cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", [
        (101, 1, 'COMPLETED', 5400.0),
        (102, 1, 'PENDING', 10000.0),
        (103, 2, 'COMPLETED', 1200.5),
        (104, 3, 'COMPLETED', 8900.0)
    ])
    conn.commit()
    return conn

# Schema string to feed to the LLM
SCHEMA_INFO = """
Table: customers (id INTEGER PRIMARY KEY, name TEXT, tier TEXT, spent REAL)
Table: orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, status TEXT, total REAL, FOREIGN KEY(customer_id) REFERENCES customers(id))
"""

# ==============================================================================
# STRUCTURED CRITIC CONTRACT (Pydantic)
# ==============================================================================
class SqlCritique(BaseModel):
    is_valid: bool = Field(description="True if SQL meets all prompt constraints and has no syntax errors.")
    status_code: str = Field(description="'APPROVED' if perfect, otherwise 'REVISE'.")
    feedback: list[str] = Field(description="Bulleted specific critiques or suggestions for the producer.")

# ==============================================================================
# REFLECTION LOOP (Producer -> Critic -> Refinement)
# ==============================================================================
def reflective_sql_generator(natural_language_query: str, schema_info: str, max_iter=3) -> str:
    print(f"--- Generating SQL for: '{natural_language_query}' ---")
    current_sql = ""
    history_feedback = ""

    for iteration in range(max_iter):
        print(f"\n[Iteration {iteration + 1}]")

        # STAGE 1: PRODUCER
        producer_prompt = f"""
        You are an expert SQLite developer.
        Schema:\n{schema_info}
        Intent:\n{natural_language_query}
        {f"Feedback to fix from previous attempt:\n{history_feedback}" if history_feedback else ""}
        Output ONLY the raw SQLite query. Do not wrap in markdown fences (no ```sql). Do not explain.
        """

        response = client.models.generate_content(
            model=MODEL_ID,
            contents=producer_prompt
        )
        current_sql = response.text.strip().replace('```sql', '').replace('```', '').strip()
        print(f"Producer generated: {current_sql}")

        # STAGE 2: CRITIC
        critic_prompt = f"""
        You are a Senior Database Architect reviewing SQLite.
        Query: {current_sql}
        Schema: {schema_info}
        Intent: {natural_language_query}

        Evaluate the query. Does it answer the intent? Is the syntax valid for SQLite?
        """

        critic_response = client.models.generate_content(
            model=MODEL_ID,
            contents=critic_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=SqlCritique,
            )
        )

        critique_dict = json.loads(critic_response.text)
        status_code = critique_dict.get('status_code', 'REVISE')
        is_valid = critique_dict.get('is_valid', False)
        feedback = critique_dict.get('feedback', [])

        print(f"Critic Status: {status_code} | Valid: {is_valid}")
        if feedback:
            print(f"Critic Feedback: {feedback}")

        # STAGE 3: LOOP DECISION
        if status_code == 'APPROVED' and is_valid:
            print(">> Query Approved by Critic!")
            return current_sql
        else:
            history_feedback = "\n".join(feedback)

    print(">> Max iterations reached. Returning best effort SQL.")
    return current_sql

# ==============================================================================
# THREE-PILLAR EXCEPTION HANDLING AND RECOVERY CASCADE
# ==============================================================================
def execute_resilient_query(conn, sql_query: str, original_intent: str, schema_info: str):
    cursor = conn.cursor()

    # Pillar 1: Detection
    try:
        cursor.execute(sql_query)
        results = cursor.fetchall()
        columns = [description[0] for description in cursor.description]
        return {"status": "SUCCESS", "columns": columns, "data": results}

    except (sqlite3.OperationalError, sqlite3.DatabaseError) as e:
        error_msg = str(e)
        print(f"\n[!] PILLAR 1 ALERT: Primary query failed: {error_msg}")

        # Pillar 2: Handling & Dynamic RCA
        rca_prompt = f"""
        The following SQLite query failed with error: '{error_msg}'
        Query: {sql_query}
        Intent: {original_intent}
        Schema: {schema_info}

        1. Provide a 1-sentence Root Cause Analysis (rca).
        2. Provide a completely safe, simplified 'fallback_sql' query (e.g. SELECT * FROM main_table LIMIT 5).
        Format exactly as JSON: {{"rca": "...", "fallback_sql": "..."}}
        """

        try:
            rca_response = client.models.generate_content(
                model=MODEL_ID,
                contents=rca_prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json")
            )
            rca_data = json.loads(rca_response.text)

            print(f"[+] PILLAR 2 RCA: {rca_data.get('rca')}")
            fallback_sql = rca_data.get('fallback_sql')
            print(f"[+] PILLAR 2 FALLBACK SQL: {fallback_sql}")

            cursor.execute(fallback_sql)
            results = cursor.fetchall()
            columns = [description[0] for description in cursor.description]
            return {
                "status": "RECOVERED",
                "columns": columns,
                "data": results,
                "rca": rca_data.get('rca')
            }

        except Exception as inner_e:
            # Pillar 3: Final Recovery / Graceful Apology
            print(f"[-] PILLAR 3 ALERT: Complete Fallback failed. {str(inner_e)}")
            return {
                "status": "FAILED",
                "message": "We couldn't retrieve the specific data you asked for, and our backup systems are temporarily disrupted. Please try again in a moment."
            }

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    db_conn = setup_ecommerce_db()

    # Test 1: Successful Reflection & Execution
    intent_1 = "Show me the names of Enterprise customers who have spent more than 10000."
    generated_sql_1 = reflective_sql_generator(intent_1, SCHEMA_INFO)
    result_1 = execute_resilient_query(db_conn, generated_sql_1, intent_1, SCHEMA_INFO)
    print("\nFinal Output Test 1:")
    print(json.dumps(result_1, indent=2))

    print("\n" + "="*60 + "\n")

    # Test 2: Forcing an Error to Trigger Resilience Cascade
    # We purposefully sabotage the SQL string to trigger the exception handler
    intent_2 = "Show all orders."
    sabotaged_sql = "SELECT * FROM non_existent_table WHERE order_id = 101;"
    print(f"--- Simulating Database Error for: '{intent_2}' ---")
    print(f"Executing invalid SQL: {sabotaged_sql}")

    result_2 = execute_resilient_query(db_conn, sabotaged_sql, intent_2, SCHEMA_INFO)
    print("\nFinal Output Test 2 (Resilient Recovery):")
    print(json.dumps(result_2, indent=2))

--- Generating SQL for: 'Show me the names of Enterprise customers who have spent more than 10000.' ---

[Iteration 1]
Producer generated: SELECT name FROM customers WHERE tier = 'Enterprise' AND spent > 10000;
Critic Status: APPROVED | Valid: True
>> Query Approved by Critic!

Final Output Test 1:
{
  "status": "SUCCESS",
  "columns": [
    "name"
  ],
  "data": [
    [
      "Alice Corp"
    ]
  ]
}


--- Simulating Database Error for: 'Show all orders.' ---
Executing invalid SQL: SELECT * FROM non_existent_table WHERE order_id = 101;

[!] PILLAR 1 ALERT: Primary query failed: no such table: non_existent_table
[+] PILLAR 2 RCA: The query failed because it referenced 'non_existent_table' which does not exist in the database schema instead of the 'orders' table.
[+] PILLAR 2 FALLBACK SQL: SELECT * FROM orders LIMIT 5;

Final Output Test 2 (Resilient Recovery):
{
  "status": "RECOVERED",
  "columns": [
    "order_id",
    "customer_id",
    "status",
    "total"
  ],
  "data": [
    [
 